In [ ]:
# 0. Setup
!pip install -q scikit-image opencv-python-headless openpyxl xlsxwriter "timm==0.6.13"

import os, re, sys, json, shutil, zipfile, warnings, platform, itertools
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.optimize import brentq
import cv2
from PIL import Image
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MODEL_LABELS = {
    "flux2pro": "FLUX.2 Pro",
    "fireflyimage5": "Firefly Image 5",
    "gemini3nanobananapro": "Nano Banana Pro",
    "gptimage2.5sunburst": "GPT Image 2.5",
    "runwayg-4image": "Runway Gen-4 Image",
}
SPACE_LABELS = {"lounge": "Lounge", "corridor": "Corridor", "patientroom": "Patient Room"}
MODEL_ORDER = list(MODEL_LABELS)
SPACE_ORDER = list(SPACE_LABELS)
EXPECTED_N_PER_CELL = 20

TARGET_SIZE = 1024
THRESH_METHODS = ("otsu", "adaptive")
ADAPTIVE_BLOCK_SIZE = 35
PRIMARY_D = "D_otsu"
ALPHA = 0.05

DV_LABELS = {
    "D_otsu": "Fractal dimension D (Otsu)",
    "D_adaptive": "Fractal dimension D (adaptive)",
    "CF": "Colorfulness (Hasler–Süsstrunk)",
    "CF_near": "Colorfulness – near region",
    "CF_mid": "Colorfulness – middle region",
    "CF_far": "Colorfulness – far region",
    "DPI": "Depth Perception Index",
    "VCI_baseline": "Visual Comfort Index (baseline)",
}
EFFECT_LABELS = {"model": "Model", "space_type": "Space type", "model:space_type": "Model × Space type"}

sns.set_theme(style="whitegrid", context="paper", font_scale=1.25)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "DejaVu Sans",
})
MODEL_PALETTE = dict(zip([MODEL_LABELS[m] for m in MODEL_ORDER],
                         sns.color_palette("colorblind", len(MODEL_ORDER))))

BASE_DIR = Path("/content/visual_comfort_analysis") if IN_COLAB else Path.cwd() / "visual_comfort_analysis"
DIRS = {
    "raw": BASE_DIR / "data/raw",
    "zips": BASE_DIR / "data/zips",
    "processed": BASE_DIR / "data/processed",
    "depth": BASE_DIR / "data/depth_maps",
    "tables": BASE_DIR / "outputs/tables",
    "figures": BASE_DIR / "outputs/figures",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def model_label(m):
    return MODEL_LABELS.get(m, m)

def space_label(s):
    return SPACE_LABELS.get(s, s)

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    plt.show()
    plt.close(fig)

def flatten_for_excel(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join(str(c) for c in col if str(c) != "") for col in df.columns]
    if isinstance(df.index, pd.MultiIndex) or df.index.name is not None:
        df = df.reset_index()
    for col in list(df.columns):
        s = df[col].dropna()
        if len(s) and isinstance(s.iloc[0], (list, tuple, np.ndarray)):
            width = max(len(v) for v in s)
            names = ([f"{col}_lower", f"{col}_upper"] if width == 2
                     else [f"{col}_{i + 1}" for i in range(width)])
            expanded = df[col].apply(
                lambda v: list(v) + [np.nan] * (width - len(v))
                if isinstance(v, (list, tuple, np.ndarray)) else [np.nan] * width)
            df[names] = pd.DataFrame(expanded.tolist(), index=df.index)
            df = df.drop(columns=[col])
    for col in df.columns:
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].astype(str)
    return df.replace([np.inf, -np.inf], np.nan)

def safe_sheet_name(name, used):
    base = re.sub(r"[\[\]\:\*\?\/\\]", "_", str(name))[:31]
    cand, i = base, 1
    while cand.lower() in used:
        suffix = f"_{i}"
        cand = base[:31 - len(suffix)] + suffix
        i += 1
    used.add(cand.lower())
    return cand

def write_excel(sheets, path):
    used = set()
    with pd.ExcelWriter(path, engine="xlsxwriter",
                        engine_kwargs={"options": {"nan_inf_to_errors": True}}) as writer:
        hdr = writer.book.add_format({"bold": True, "bg_color": "#DDE6F0", "border": 1,
                                      "text_wrap": True, "valign": "top"})
        for name, df in sheets.items():
            if df is None or len(df) == 0:
                continue
            clean = flatten_for_excel(df)
            sname = safe_sheet_name(name, used)
            clean.to_excel(writer, sheet_name=sname, index=False)
            ws = writer.sheets[sname]
            for j, col in enumerate(clean.columns):
                ws.write(0, j, str(col), hdr)
                longest = clean[col].astype(str).str.len().max() if len(clean) else 0
                ws.set_column(j, j, int(min(45, max(10, len(str(col)) + 2, longest + 2))))
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, len(clean), max(0, len(clean.columns) - 1))
    return path

print(f"Device: {DEVICE} | torch {torch.__version__}")
print("Working directory:", BASE_DIR)

In [ ]:
# 1. Image upload and design check
IMG_EXT = (".png", ".jpg", ".jpeg")
FNAME_PATTERN = re.compile(
    r"^(?P<model>[a-z0-9.\-]+)_(?P<space>" + "|".join(SPACE_ORDER) + r")_(?P<rep>\d+)$",
    re.IGNORECASE,
)
DRIVE_ZIP_PATH = None

def normalize_filename(fname):
    name = Path(str(fname)).name.strip()
    stem, ext = os.path.splitext(name)
    if ext.lower() not in IMG_EXT + (".zip",):
        stem, ext = name, ""
    stem = re.sub(r"\s*\(\d+\)$", "", stem)
    stem = re.sub(r"\s+", "", stem).lower()
    return stem, ext.lower()

def canonical_id(stem):
    m = FNAME_PATTERN.match(stem.strip().lower())
    if not m:
        return None, None
    model, space, rep = m.group("model"), m.group("space").lower(), int(m.group("rep"))
    return f"{model}_{space}_{rep:02d}", (model, space, rep)

def _store_image(name, data):
    stem, ext = normalize_filename(name)
    if ext not in IMG_EXT or stem.startswith("."):
        return None
    dest = DIRS["raw"] / f"{stem}{ext}"
    dest.write_bytes(data)
    return dest.name

def _extract_zip(zpath):
    n_saved, skipped = 0, []
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            base = Path(info.filename).name
            if info.is_dir() or "__MACOSX" in info.filename or base.startswith("."):
                continue
            if _store_image(base, zf.read(info)):
                n_saved += 1
            else:
                skipped.append(info.filename)
    return n_saved, skipped

def upload_images():
    if not IN_COLAB:
        print("Not running in Colab. Copy the images to:", DIRS["raw"])
        return
    if DRIVE_ZIP_PATH:
        from google.colab import drive
        drive.mount("/content/drive")
        n_saved, skipped = _extract_zip(DRIVE_ZIP_PATH)
    else:
        print("Select the image files or a .zip archive containing them...")
        uploaded = files.upload()
        n_saved, skipped = 0, []
        for fname, content in uploaded.items():
            if fname.lower().endswith(".zip"):
                zpath = DIRS["zips"] / Path(fname).name
                zpath.write_bytes(content)
                n, s = _extract_zip(zpath)
                n_saved, skipped = n_saved + n, skipped + s
            elif _store_image(fname, content):
                n_saved += 1
            else:
                skipped.append(fname)
    print(f"{n_saved} images saved.")
    if skipped:
        print(f"{len(skipped)} non-image files skipped:", skipped[:10])

def build_manifest():
    rows, rejected = [], []
    for p in sorted(DIRS["raw"].iterdir()):
        if p.suffix.lower() not in IMG_EXT:
            continue
        cid, parts = canonical_id(p.stem)
        if cid is None:
            rejected.append(p.name)
            continue
        try:
            with Image.open(p) as im:
                w, h = im.size
        except Exception:
            rejected.append(p.name + " (unreadable)")
            continue
        model, space, rep = parts
        rows.append({
            "image_id": cid, "file": p.name, "model": model, "space_type": space,
            "replicate": rep, "model_label": model_label(model), "space_label": space_label(space),
            "file_format": p.suffix.lower().lstrip(".").replace("jpeg", "jpg"),
            "orig_width": w, "orig_height": h, "source_size": p.stat().st_size, "raw_path": str(p),
        })
    man = pd.DataFrame(rows)
    if rejected:
        print(f"WARNING: {len(rejected)} files do not follow the naming scheme and were excluded:")
        for r in rejected:
            print("   -", r)
    if len(man) == 0:
        return man
    dup = man[man.duplicated("image_id", keep=False)]
    if len(dup):
        print("WARNING: duplicate model/space/replicate identifiers (PNG kept):")
        print(dup[["image_id", "file"]].to_string(index=False))
        man = (man.assign(_png=(man.file_format != "png").astype(int))
                  .sort_values(["image_id", "_png"]).drop_duplicates("image_id").drop(columns="_png"))
    unknown = sorted(set(man.model) - set(MODEL_ORDER))
    if unknown:
        print("WARNING: model tags not defined in MODEL_LABELS:", unknown)
    return man.sort_values(["model", "space_type", "replicate"]).reset_index(drop=True)

def design_check(man):
    models = [m for m in MODEL_ORDER if m in set(man.model)] + sorted(set(man.model) - set(MODEL_ORDER))
    counts = (pd.crosstab(man.model, man.space_type)
                .reindex(index=models, columns=SPACE_ORDER, fill_value=0))
    counts["total"] = counts.sum(axis=1)
    reps = (man.groupby(["model", "space_type"])["replicate"]
               .agg(n="count", rep_min="min", rep_max="max").reset_index())
    reps["missing_in_range"] = [
        ", ".join(str(r) for r in sorted(set(range(a, b + 1)) - set(
            man.loc[(man.model == m) & (man.space_type == s), "replicate"]))) or "-"
        for m, s, a, b in zip(reps.model, reps.space_type, reps.rep_min, reps.rep_max)]
    reps["n_ok"] = reps["n"] == EXPECTED_N_PER_CELL
    fmt = pd.crosstab(man.model, man.file_format).reindex(index=models, fill_value=0)
    res_rows = []
    for m in models:
        g = man[man.model == m]
        res_rows.append({
            "model": m,
            "median_width": g.orig_width.median(), "median_height": g.orig_height.median(),
            "min_width": g.orig_width.min(), "max_width": g.orig_width.max(),
            "n_distinct_sizes": g[["orig_width", "orig_height"]].drop_duplicates().shape[0],
            "share_non_square": float((g.orig_width != g.orig_height).mean()),
        })
    res = pd.DataFrame(res_rows)
    warnings_list = []
    if len(reps[~reps.n_ok]):
        warnings_list.append(f"{len(reps[~reps.n_ok])} cells do not contain {EXPECTED_N_PER_CELL} images.")
    png_share = man.assign(p=(man.file_format == "png")).groupby("model")["p"].mean()
    if png_share.nunique() > 1 and png_share.isin([0.0, 1.0]).any():
        warnings_list.append("File format is confounded with model.")
    if res["median_width"].nunique() > 1:
        warnings_list.append("Native resolution differs between models.")
    if (res["share_non_square"] > 0).any():
        warnings_list.append("Non-square images found. They will be center-cropped.")
    return counts.reset_index(), reps, fmt.reset_index(), res, warnings_list

upload_images()
manifest = build_manifest()
if len(manifest) == 0:
    raise RuntimeError("No valid images found.")

design_counts, design_reps, design_format, design_resolution, design_warnings = design_check(manifest)
manifest.drop(columns=["raw_path"]).to_csv(DIRS["tables"] / "manifest.csv", index=False)

print(f"\nValid images: {len(manifest)}")
display(design_counts)
display(design_format)
display(design_resolution)
if len(design_reps[~design_reps.n_ok]):
    display(design_reps[~design_reps.n_ok])
for w in design_warnings:
    print("WARNING:", w)

In [ ]:
# 2. Preprocessing
def read_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Cannot read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def standardize_image(img, size=TARGET_SIZE):
    h, w = img.shape[:2]
    s = min(h, w)
    top, left = (h - s) // 2, (w - s) // 2
    return cv2.resize(img[top:top + s, left:left + s], (size, size), interpolation=cv2.INTER_AREA)

processed_paths = []
for row in manifest.itertuples():
    out = DIRS["processed"] / f"{row.image_id}.png"
    if not out.exists() or out.stat().st_mtime < Path(row.raw_path).stat().st_mtime:
        std = standardize_image(read_rgb(row.raw_path))
        cv2.imwrite(str(out), cv2.cvtColor(std, cv2.COLOR_RGB2BGR))
    processed_paths.append(str(out))
manifest["processed_path"] = processed_paths
print(f"{len(manifest)} images standardized to {TARGET_SIZE}x{TARGET_SIZE} PNG.")

models_present = [m for m in MODEL_ORDER if m in set(manifest.model)] + \
                 sorted(set(manifest.model) - set(MODEL_ORDER))
spaces_present = [s for s in SPACE_ORDER if s in set(manifest.space_type)]
fig, axes = plt.subplots(len(models_present), len(spaces_present),
                         figsize=(3.2 * len(spaces_present), 3.2 * len(models_present)), squeeze=False)
for i, m in enumerate(models_present):
    for j, s in enumerate(spaces_present):
        ax = axes[i, j]
        cell = manifest[(manifest.model == m) & (manifest.space_type == s)].sort_values("replicate")
        if len(cell):
            ax.imshow(read_rgb(cell.iloc[0].processed_path))
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if i == 0:
            ax.set_title(space_label(s), fontsize=13)
        if j == 0:
            ax.set_ylabel(model_label(m), fontsize=12)
fig.tight_layout()
save_fig(fig, "figure01_example_images")

In [ ]:
# 3a. Image metrics
from skimage import color as skcolor, filters, morphology

BOX_SIZES = (2, 4, 8, 16, 32, 64)

def preprocess_for_fractal(image_rgb, method="otsu"):
    gray = skcolor.rgb2gray(image_rgb)
    if method == "otsu":
        thresh = filters.threshold_otsu(gray)
    elif method == "adaptive":
        thresh = filters.threshold_local(gray, block_size=ADAPTIVE_BLOCK_SIZE)
    else:
        raise ValueError("method must be 'otsu' or 'adaptive'")
    return morphology.skeletonize(gray < thresh)

def _box_count(binary, box):
    S0, S1 = binary.shape
    c = binary[:S0 - S0 % box, :S1 - S1 % box]
    return int(c.reshape(c.shape[0] // box, box, c.shape[1] // box, box).any(axis=(1, 3)).sum())

def fractal_dimension(binary, box_sizes=BOX_SIZES):
    counts = [_box_count(binary, s) for s in box_sizes]
    valid = [(s, c) for s, c in zip(box_sizes, counts) if c > 0]
    if len(valid) < 3:
        return np.nan, np.nan
    sizes, cnts = zip(*valid)
    ls, lc = np.log(sizes), np.log(cnts)
    coef = np.polyfit(ls, lc, 1)
    pred = np.polyval(coef, ls)
    ss_tot = np.sum((lc - lc.mean()) ** 2)
    r2 = 1 - np.sum((lc - pred) ** 2) / ss_tot if ss_tot > 0 else np.nan
    return float(-coef[0]), float(r2)

def colorfulness_hasler_susstrunk(image_rgb):
    img = image_rgb.astype(float)
    R, G, B = img[..., 0], img[..., 1], img[..., 2]
    rg, yb = R - G, 0.5 * (R + G) - B
    return float(np.sqrt(np.std(rg) ** 2 + np.std(yb) ** 2) + 0.3 * np.sqrt(np.mean(rg) ** 2 + np.mean(yb) ** 2))

def normalize01(x):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def region_colorfulness(image_rgb, inv_depth_norm):
    q1, q2 = np.quantile(inv_depth_norm, [1 / 3, 2 / 3])
    masks = {"CF_far": inv_depth_norm <= q1,
             "CF_mid": (inv_depth_norm > q1) & (inv_depth_norm <= q2),
             "CF_near": inv_depth_norm > q2}
    return {k: (colorfulness_hasler_susstrunk(image_rgb[m].reshape(1, -1, 3)) if m.sum() > 0 else np.nan)
            for k, m in masks.items()}

print("Metric functions loaded.")

In [ ]:
# 3b. Depth model (MiDaS)
def load_midas():
    preferred = ["DPT_Hybrid", "MiDaS_small"] if DEVICE == "cuda" else ["MiDaS_small", "DPT_Hybrid"]
    for mtype in preferred:
        try:
            model = torch.hub.load("intel-isl/MiDaS", mtype, trust_repo=True)
            tr = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True)
            transform = tr.dpt_transform if "DPT" in mtype else tr.small_transform
            model.to(DEVICE).eval()
            print(f"MiDaS model loaded: {mtype} ({DEVICE})")
            return model, transform, mtype
        except Exception as e:
            print(f"Could not load {mtype}: {e}")
    raise RuntimeError("MiDaS could not be loaded.")

midas, midas_transform, MIDAS_MODEL_TYPE = load_midas()

def extract_depth_map(image_rgb):
    batch = midas_transform(image_rgb).to(DEVICE)
    with torch.no_grad():
        pred = midas(batch)
        pred = torch.nn.functional.interpolate(pred.unsqueeze(1), size=image_rgb.shape[:2],
                                               mode="bicubic", align_corners=False).squeeze()
    return pred.cpu().numpy()

In [ ]:
# 3c. Metric extraction
CACHE_PATH = DIRS["tables"] / "image_metrics_cache.csv"
CACHE_KEY = ["image_id", "file", "source_size"]
cache = (pd.read_csv(CACHE_PATH) if CACHE_PATH.exists()
         else pd.DataFrame(columns=CACHE_KEY + ["midas_model"]))
cache = cache[cache["midas_model"] == MIDAS_MODEL_TYPE].merge(manifest[CACHE_KEY], on=CACHE_KEY, how="inner")
todo = manifest[~manifest.image_id.isin(set(cache.image_id))]
print(f"Cached: {len(cache)} | to process: {len(todo)}")

new_rows = []
def _flush():
    global cache
    if new_rows:
        cache = (pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
                   .drop_duplicates("image_id", keep="last"))
        new_rows.clear()
        cache.to_csv(CACHE_PATH, index=False)

t0 = datetime.now()
for i, row in enumerate(todo.itertuples(), 1):
    img = read_rgb(row.processed_path)
    rec = {"image_id": row.image_id, "file": row.file, "source_size": row.source_size,
           "midas_model": MIDAS_MODEL_TYPE}
    for method in THRESH_METHODS:
        skel = preprocess_for_fractal(img, method)
        D, r2 = fractal_dimension(skel)
        rec[f"D_{method}"], rec[f"R2_{method}"] = D, r2
        rec[f"skeleton_share_{method}"] = float(skel.mean())
    inv = normalize01(extract_depth_map(img))
    rec["DPI"] = float(np.std(inv))
    rec["CF"] = colorfulness_hasler_susstrunk(img)
    rec.update(region_colorfulness(img, inv))
    cv2.imwrite(str(DIRS["depth"] / f"{row.image_id}.png"), (inv * 255).astype(np.uint8))
    new_rows.append(rec)
    if i % 25 == 0:
        _flush()
        print(f"  {i}/{len(todo)} processed ({(datetime.now() - t0).seconds} s)")
_flush()

metrics_df = cache[cache.image_id.isin(set(manifest.image_id))].drop(columns=["file", "source_size"]).copy()
metrics_df["low_fit_quality"] = (metrics_df["R2_otsu"] < 0.95) | (metrics_df["R2_adaptive"] < 0.95)
print(f"Metrics computed for {len(metrics_df)} images. Images with R2 < 0.95: {int(metrics_df.low_fit_quality.sum())}")

qc = manifest.iloc[0]
qc_img = read_rgb(qc.processed_path)
qc_inv = cv2.imread(str(DIRS["depth"] / f"{qc.image_id}.png"), cv2.IMREAD_GRAYSCALE)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
axes[0].imshow(qc_img); axes[0].set_title("Standardized image")
im = axes[1].imshow(qc_inv, cmap="inferno"); axes[1].set_title("MiDaS relative inverse depth")
fig.colorbar(im, ax=axes[1], fraction=0.046, label="near (high) – far (low)")
axes[2].imshow(preprocess_for_fractal(qc_img, "otsu"), cmap="gray_r"); axes[2].set_title("Skeleton – Otsu")
axes[3].imshow(preprocess_for_fractal(qc_img, "adaptive"), cmap="gray_r"); axes[3].set_title("Skeleton – adaptive")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "figure02_processing_pipeline")

In [ ]:
# 4. Image-level dataset and Visual Comfort Index
master_df = manifest.merge(metrics_df, on="image_id", how="inner")

def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

master_df["CF_norm"] = minmax(master_df["CF"])
master_df["DPI_norm"] = minmax(master_df["DPI"])
master_df["VCI_baseline"] = 0.5 * master_df["CF_norm"] + 0.5 * master_df["DPI_norm"]

model_levels = [m for m in MODEL_ORDER if m in set(master_df.model)] + \
               sorted(set(master_df.model) - set(MODEL_ORDER))
space_levels = [s for s in SPACE_ORDER if s in set(master_df.space_type)]
master_df["model"] = pd.Categorical(master_df["model"], categories=model_levels, ordered=True)
master_df["space_type"] = pd.Categorical(master_df["space_type"], categories=space_levels, ordered=True)
MODEL_LABEL_ORDER = [model_label(m) for m in model_levels]
SPACE_LABEL_ORDER = [space_label(s) for s in space_levels]
master_df = master_df.sort_values(["model", "space_type", "replicate"]).reset_index(drop=True)

master_cols = ["image_id", "model", "model_label", "space_type", "space_label", "replicate", "file",
               "file_format", "orig_width", "orig_height", "D_otsu", "R2_otsu", "D_adaptive", "R2_adaptive",
               "skeleton_share_otsu", "skeleton_share_adaptive", "low_fit_quality", "DPI", "CF",
               "CF_near", "CF_mid", "CF_far", "CF_norm", "DPI_norm", "VCI_baseline", "midas_model"]
master_df = master_df[master_cols]
write_excel({"image_level_data": master_df}, DIRS["tables"] / "master_results.xlsx")
master_df.to_csv(DIRS["tables"] / "master_results.csv", index=False)

print(f"Dataset: {master_df.shape[0]} images x {master_df.shape[1]} variables")
display(master_df.groupby(["model", "space_type"], observed=True)[[PRIMARY_D, "CF", "DPI", "VCI_baseline"]]
        .mean().round(3))

In [ ]:
# 5. Statistical functions
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    n = len(pv)
    if n == 0:
        return out
    adj = np.empty(n)
    running = 0.0
    for rank, idx in enumerate(np.argsort(pv)):
        running = max(running, (n - rank) * pv[idx])
        adj[idx] = min(1.0, running)
    out[ok] = adj
    return out

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "p = n/a"
    return "p < .001" if p < 0.001 else "p = " + f"{p:.3f}".replace("0.", ".", 1)

def hedges_g(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    d = (np.mean(x) - np.mean(y)) / sp if sp > 0 else np.nan
    return d * (1 - 3 / (4 * (nx + ny) - 9))

def describe(df, dvs, by):
    rows = []
    for keys, g in df.groupby(by, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        for dv in dvs:
            x = g[dv].dropna().to_numpy(float)
            n = len(x)
            m = x.mean() if n else np.nan
            sd = x.std(ddof=1) if n > 1 else np.nan
            se = sd / np.sqrt(n) if n > 1 else np.nan
            tc = stats.t.ppf(0.975, n - 1) if n > 1 else np.nan
            rec = dict(zip(by, [str(k) for k in keys]))
            rec.update({"variable": dv, "n": n, "mean": m, "sd": sd, "se": se,
                        "ci95_lower": m - tc * se, "ci95_upper": m + tc * se,
                        "median": np.median(x) if n else np.nan,
                        "q1": np.percentile(x, 25) if n else np.nan,
                        "q3": np.percentile(x, 75) if n else np.nan,
                        "min": x.min() if n else np.nan, "max": x.max() if n else np.nan})
            rows.append(rec)
    out = pd.DataFrame(rows)
    if "model" in by:
        out.insert(by.index("model") + 1, "model_label", out["model"].map(model_label))
    if "space_type" in by:
        out.insert(list(out.columns).index("space_type") + 1, "space_label", out["space_type"].map(space_label))
    return out

def power_f(f, N, df1, df2, alpha=ALPHA):
    if f is None or np.isnan(f) or df2 <= 0 or df1 <= 0:
        return np.nan
    crit = stats.f.ppf(1 - alpha, df1, df2)
    return float(stats.ncf.sf(crit, df1, df2, f ** 2 * N))

def required_total_n(f, df1, n_groups, target=0.80, alpha=ALPHA, n_max=100000):
    N = n_groups + 2
    while N <= n_max:
        if power_f(f, N, df1, N - n_groups, alpha) >= target:
            return N
        N += 1
    return np.nan

def min_detectable_f(N, df1, df2, target=0.80, alpha=ALPHA):
    try:
        return float(brentq(lambda f: power_f(f, N, df1, df2, alpha) - target, 1e-4, 5))
    except ValueError:
        return np.nan

def f_from_eta2(eta2):
    if eta2 is None or np.isnan(eta2):
        return np.nan
    eta2 = min(max(eta2, 0.0), 0.999)
    return float(np.sqrt(eta2 / (1 - eta2)))

def power_paired_t(dz, n, alpha=ALPHA):
    if np.isnan(dz) or n < 3:
        return np.nan
    df = n - 1
    nc = abs(dz) * np.sqrt(n)
    tc = stats.t.ppf(1 - alpha / 2, df)
    return float(stats.nct.sf(tc, df, nc) + stats.nct.cdf(-tc, df, nc))

def power_corr(r, n, alpha=ALPHA):
    if n <= 3 or np.isnan(r):
        return np.nan
    z = np.arctanh(min(abs(r), 0.999999)) * np.sqrt(n - 3)
    zc = stats.norm.ppf(1 - alpha / 2)
    return float(stats.norm.sf(zc - z) + stats.norm.cdf(-zc - z))

def welch_anova(groups):
    k = len(groups)
    n = np.array([len(g) for g in groups], float)
    m = np.array([np.mean(g) for g in groups])
    v = np.array([np.var(g, ddof=1) for g in groups])
    if np.any(v <= 0):
        return np.nan, np.nan, np.nan, np.nan
    w = n / v
    mw = np.sum(w * m) / np.sum(w)
    A = np.sum(w * (m - mw) ** 2) / (k - 1)
    tmp = np.sum((1 - w / np.sum(w)) ** 2 / (n - 1))
    F = A / (1 + 2 * (k - 2) / (k ** 2 - 1) * tmp)
    df1, df2 = k - 1, (k ** 2 - 1) / (3 * tmp)
    return float(F), float(df1), float(df2), float(stats.f.sf(F, df1, df2))

def oneway_tests(df, dv, group, order):
    sub = df[[dv, group]].dropna()
    levels = [l for l in order if (sub[group] == l).sum() > 0]
    groups = [sub.loc[sub[group] == l, dv].to_numpy(float) for l in levels]
    k, N = len(groups), sum(len(g) for g in groups)
    if k < 2:
        return None
    sh = [stats.shapiro(g).pvalue if len(g) >= 3 else np.nan for g in groups]
    all_normal = all(p > ALPHA for p in sh if not np.isnan(p))
    bf = stats.levene(*groups, center="median")
    F, pF = stats.f_oneway(*groups)
    allv = np.concatenate(groups)
    grand = allv.mean()
    ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    sst = ((allv - grand) ** 2).sum()
    msw = (sst - ssb) / (N - k)
    eta2 = ssb / sst if sst > 0 else np.nan
    omega2 = (ssb - (k - 1) * msw) / (sst + msw) if sst > 0 else np.nan
    Fw, df1w, df2w, pw = welch_anova(groups)
    H, pH = stats.kruskal(*groups)
    eps2 = H * (N + 1) / (N ** 2 - 1)
    primary = "welch_anova" if all_normal and not np.isnan(pw) else "kruskal_wallis"
    return {
        "dv": dv, "factor": group, "k": k, "N": N,
        "min_shapiro_p": np.nanmin(sh), "all_groups_normal": all_normal,
        "brown_forsythe_F": float(bf.statistic), "brown_forsythe_p": float(bf.pvalue),
        "anova_F": float(F), "anova_df1": k - 1, "anova_df2": N - k, "anova_p": float(pF),
        "eta2": eta2, "omega2": omega2,
        "welch_F": Fw, "welch_df1": df1w, "welch_df2": df2w, "welch_p": pw,
        "kruskal_H": float(H), "kruskal_df": k - 1, "kruskal_p": float(pH), "epsilon2": eps2,
        "primary_test": primary, "p_primary": pw if primary == "welch_anova" else float(pH),
        "achieved_power": power_f(f_from_eta2(eta2), N, k - 1, N - k),
    }

def games_howell(df, dv, group, order):
    sub = df[[dv, group]].dropna()
    gs = {l: sub.loc[sub[group] == l, dv].to_numpy(float) for l in order if (sub[group] == l).sum() > 1}
    k = len(gs)
    rows = []
    for a, b in itertools.combinations(gs, 2):
        x, y = gs[a], gs[b]
        na, nb = len(x), len(y)
        va, vb = x.var(ddof=1) / na, y.var(ddof=1) / nb
        diff, se = x.mean() - y.mean(), np.sqrt(va + vb)
        t = diff / se
        dfw = (va + vb) ** 2 / (va ** 2 / (na - 1) + vb ** 2 / (nb - 1))
        p = float(stats.studentized_range.sf(abs(t) * np.sqrt(2), k, dfw))
        qc = stats.studentized_range.ppf(0.95, k, dfw) / np.sqrt(2)
        rows.append({"A": str(a), "B": str(b), "mean_A": x.mean(), "mean_B": y.mean(), "mean_diff": diff,
                     "se": se, "t": t, "df": dfw, "p_adj": min(p, 1.0),
                     "diff_ci95_lower": diff - qc * se, "diff_ci95_upper": diff + qc * se,
                     "effect_size": hedges_g(x, y), "effect_size_type": "hedges_g",
                     "method": "Games-Howell"})
    return pd.DataFrame(rows)

def dunn_test(df, dv, group, order):
    sub = df[[dv, group]].dropna().copy()
    sub["_r"] = stats.rankdata(sub[dv])
    N = len(sub)
    _, cnt = np.unique(sub[dv], return_counts=True)
    s2 = N * (N + 1) / 12 - np.sum(cnt ** 3 - cnt) / (12 * (N - 1))
    levels = [l for l in order if (sub[group] == l).sum() > 0]
    rows = []
    for a, b in itertools.combinations(levels, 2):
        ra, rb = sub.loc[sub[group] == a, "_r"], sub.loc[sub[group] == b, "_r"]
        z = (ra.mean() - rb.mean()) / np.sqrt(s2 * (1 / len(ra) + 1 / len(rb)))
        x = sub.loc[sub[group] == a, dv].to_numpy(float)
        y = sub.loc[sub[group] == b, dv].to_numpy(float)
        U = stats.mannwhitneyu(x, y, alternative="two-sided").statistic
        rows.append({"A": str(a), "B": str(b), "median_A": np.median(x), "median_B": np.median(y),
                     "mean_rank_A": ra.mean(), "mean_rank_B": rb.mean(), "z": z,
                     "p_unadj": float(2 * stats.norm.sf(abs(z))),
                     "effect_size": 2 * U / (len(x) * len(y)) - 1, "effect_size_type": "rank_biserial_r",
                     "method": "Dunn (Holm)"})
    out = pd.DataFrame(rows)
    if len(out):
        out["p_adj"] = holm(out["p_unadj"])
    return out

def posthoc(df, dv, group, order, primary):
    return games_howell(df, dv, group, order) if primary == "welch_anova" else dunn_test(df, dv, group, order)

def _rss(y, X):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    return float(resid @ resid), resid, int(np.linalg.matrix_rank(X))

def _dummies(s):
    return pd.get_dummies(s.astype(str), drop_first=True, dtype=float).to_numpy()

def twoway_anova(df, dv, A, B):
    sub = df[[dv, A, B]].dropna().copy()
    y = sub[dv].to_numpy(float)
    N = len(y)
    one = np.ones((N, 1))
    XA, XB = _dummies(sub[A]), _dummies(sub[B])
    XAB = (np.column_stack([XA[:, i] * XB[:, j] for i in range(XA.shape[1]) for j in range(XB.shape[1])])
           if XA.shape[1] and XB.shape[1] else np.empty((N, 0)))
    rss_A, _, r_A = _rss(y, np.hstack([one, XA]))
    rss_B, _, r_B = _rss(y, np.hstack([one, XB]))
    rss_AB, _, r_AB = _rss(y, np.hstack([one, XA, XB]))
    rss_F, resid, r_F = _rss(y, np.hstack([one, XA, XB, XAB]))
    df_e = N - r_F
    mse = rss_F / df_e
    rows = []
    for name, ss, d in [(A, rss_B - rss_AB, r_AB - r_B), (B, rss_A - rss_AB, r_AB - r_A),
                        (f"{A}:{B}", rss_AB - rss_F, r_F - r_AB)]:
        F = (ss / d) / mse if d > 0 else np.nan
        pes = ss / (ss + rss_F)
        pom = max((ss - d * mse) / (ss + (N - d) * mse), 0.0)
        rows.append({"effect": name, "SS": ss, "df": d, "MS": ss / d if d > 0 else np.nan, "F": F,
                     "df_error": df_e, "p": float(stats.f.sf(F, d, df_e)) if d > 0 else np.nan,
                     "partial_eta2": pes, "partial_omega2": pom,
                     "achieved_power": power_f(f_from_eta2(pes), N, d, df_e)})
    rows.append({"effect": "Residual", "SS": rss_F, "df": df_e, "MS": mse})
    return {"table": pd.DataFrame(rows), "residuals": resid, "N": N}

def art_anova(df, dv, A, B):
    sub = df[[dv, A, B]].dropna().copy()
    sub[A], sub[B] = sub[A].astype(str), sub[B].astype(str)
    y = sub[dv].to_numpy(float)
    grand = y.mean()
    cell = sub.groupby([A, B])[dv].transform("mean").to_numpy()
    mA = sub.groupby(A)[dv].transform("mean").to_numpy()
    mB = sub.groupby(B)[dv].transform("mean").to_numpy()
    resid = y - cell
    aligned = {A: resid + (mA - grand), B: resid + (mB - grand),
               f"{A}:{B}": resid + (cell - mA - mB + grand)}
    rows = []
    for eff, al in aligned.items():
        tmp = sub[[A, B]].copy()
        tmp["_rank"] = stats.rankdata(al)
        tab = twoway_anova(tmp, "_rank", A, B)["table"]
        r = tab.loc[tab.effect == eff].iloc[0]
        rows.append({"effect": eff, "F": r["F"], "df": r["df"], "df_error": r["df_error"], "p": r["p"],
                     "partial_eta2_ranks": r["partial_eta2"]})
    return pd.DataFrame(rows)

def corr_with_ci(x, y, method="auto"):
    d = pd.DataFrame({"x": np.asarray(x, float), "y": np.asarray(y, float)}).dropna()
    n = len(d)
    if n < 4:
        return {"method": method, "n": n, "r": np.nan, "ci95_lower": np.nan, "ci95_upper": np.nan,
                "p": np.nan, "power": np.nan}
    if method == "auto":
        normal = stats.shapiro(d.x).pvalue > ALPHA and stats.shapiro(d.y).pvalue > ALPHA
        method = "pearson" if normal else "spearman"
    res = stats.pearsonr(d.x, d.y) if method == "pearson" else stats.spearmanr(d.x, d.y)
    r, p = float(res[0]), float(res[1])
    se = 1 / np.sqrt(n - 3) if method == "pearson" else np.sqrt((1 + r ** 2 / 2) / (n - 3))
    z, zc = np.arctanh(np.clip(r, -0.999999, 0.999999)), stats.norm.ppf(0.975)
    return {"method": method, "n": n, "r": r, "ci95_lower": float(np.tanh(z - zc * se)),
            "ci95_upper": float(np.tanh(z + zc * se)), "p": p, "power": power_corr(r, n)}

def icc_table(wide):
    X = np.asarray(wide, float)
    n, k = X.shape
    grand = X.mean()
    ss_r = k * ((X.mean(axis=1) - grand) ** 2).sum()
    ss_c = n * ((X.mean(axis=0) - grand) ** 2).sum()
    ss_e = ((X - grand) ** 2).sum() - ss_r - ss_c
    msr, msc, mse = ss_r / (n - 1), ss_c / (k - 1), ss_e / ((n - 1) * (k - 1))
    df1, df2 = n - 1, (n - 1) * (k - 1)
    F = msr / mse
    FL = F / stats.f.ppf(0.975, df1, df2)
    FU = F * stats.f.ppf(0.975, df2, df1)
    icc31 = (msr - mse) / (msr + (k - 1) * mse)
    icc3k = (msr - mse) / msr
    icc21 = (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)
    icc2k = (msr - mse) / (msr + (msc - mse) / n)
    a = k * icc21 / (n * (1 - icc21))
    b = 1 + k * icc21 * (n - 1) / (n * (1 - icc21))
    v = (a * msc + b * mse) ** 2 / ((a * msc) ** 2 / (k - 1) + (b * mse) ** 2 / ((n - 1) * (k - 1)))
    Fs, Fss = stats.f.ppf(0.975, n - 1, v), stats.f.ppf(0.975, v, n - 1)
    L21 = n * (msr - Fs * mse) / (Fs * (k * msc + (k * n - k - n) * mse) + n * msr)
    U21 = n * (Fss * msr - mse) / (k * msc + (k * n - k - n) * mse + n * Fss * msr)
    p = float(stats.f.sf(F, df1, df2))
    return pd.DataFrame([
        {"type": "ICC(2,1)", "description": "two-way random, absolute agreement, single rater", "icc": icc21,
         "ci95_lower": L21, "ci95_upper": U21},
        {"type": "ICC(2,k)", "description": "two-way random, absolute agreement, mean of k raters", "icc": icc2k,
         "ci95_lower": L21 * k / (1 + L21 * (k - 1)), "ci95_upper": U21 * k / (1 + U21 * (k - 1))},
        {"type": "ICC(3,1)", "description": "two-way mixed, consistency, single rater", "icc": icc31,
         "ci95_lower": (FL - 1) / (FL + k - 1), "ci95_upper": (FU - 1) / (FU + k - 1)},
        {"type": "ICC(3,k)", "description": "two-way mixed, consistency, mean of k raters", "icc": icc3k,
         "ci95_lower": 1 - 1 / FL, "ci95_upper": 1 - 1 / FU},
    ]).assign(F=F, df1=df1, df2=df2, p=p, n_targets=n, k_raters=k)

def front(df, cols):
    if df is None or len(df) == 0:
        return df
    cols = [c for c in cols if c in df.columns]
    return df[cols + [c for c in df.columns if c not in cols]]

print("Statistical functions loaded.")

In [ ]:
# 6. Power analysis
N_TOTAL = len(master_df)
n_models = master_df["model"].nunique()
n_spaces = master_df["space_type"].nunique()
n_cells = n_models * n_spaces
design_effects = [("model", n_models - 1), ("space_type", n_spaces - 1),
                  ("model:space_type", (n_models - 1) * (n_spaces - 1))]

apriori_rows = []
for eff, d1 in design_effects:
    for lab, f in (("small", 0.10), ("medium", 0.25), ("large", 0.40)):
        Nreq = required_total_n(f, d1, n_cells)
        apriori_rows.append({"effect": EFFECT_LABELS[eff], "df_effect": d1, "effect_size": lab, "cohens_f": f,
                             "alpha": ALPHA, "target_power": 0.80, "required_total_N": Nreq,
                             "required_n_per_cell": int(np.ceil(Nreq / n_cells)) if not np.isnan(Nreq) else np.nan,
                             "current_total_N": N_TOTAL})
power_apriori_df = pd.DataFrame(apriori_rows)

sens_rows = []
for eff, d1 in design_effects:
    for target in (0.80, 0.90):
        f_min = min_detectable_f(N_TOTAL, d1, N_TOTAL - n_cells, target)
        sens_rows.append({"effect": EFFECT_LABELS[eff], "df_effect": d1, "df_error": N_TOTAL - n_cells,
                          "N": N_TOTAL, "alpha": ALPHA, "power": target,
                          "min_detectable_cohens_f": f_min,
                          "min_detectable_partial_eta2": f_min ** 2 / (1 + f_min ** 2)})
power_sensitivity_df = pd.DataFrame(sens_rows)

display(power_apriori_df)
display(power_sensitivity_df.round(3))

In [ ]:
# 7. Descriptive statistics
DVS = [PRIMARY_D, "CF", "DPI", "VCI_baseline"]
ALL_METRICS = ["D_otsu", "D_adaptive", "CF", "CF_near", "CF_mid", "CF_far", "DPI", "VCI_baseline"]

desc_model = describe(master_df, ALL_METRICS, ["model"])
desc_space = describe(master_df, ALL_METRICS, ["space_type"])
desc_cell = describe(master_df, ALL_METRICS, ["model", "space_type"])
display(desc_model[desc_model.variable.isin(DVS)].round(3))

fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))
for ax, dv in zip(axes.flat, DVS):
    sns.pointplot(data=master_df, x="space_label", y=dv, hue="model_label", order=SPACE_LABEL_ORDER,
                  hue_order=MODEL_LABEL_ORDER, palette=MODEL_PALETTE, dodge=0.45, errorbar=("ci", 95),
                  seed=RANDOM_SEED, ax=ax)
    ax.set_title(DV_LABELS[dv]); ax.set_xlabel("Space type"); ax.set_ylabel(DV_LABELS[dv])
    if ax.get_legend() is not None:
        ax.get_legend().remove()
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.05, 1, 1])
save_fig(fig, "figure05_interaction_plots")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, dv in zip(axes, [PRIMARY_D, "VCI_baseline"]):
    piv = (master_df.pivot_table(index="model_label", columns="space_label", values=dv, aggfunc="mean")
                    .reindex(index=MODEL_LABEL_ORDER, columns=SPACE_LABEL_ORDER))
    sns.heatmap(piv, annot=True, fmt=".3f", cmap="viridis", ax=ax, cbar_kws={"label": DV_LABELS[dv]})
    ax.grid(False)
    ax.set_title(DV_LABELS[dv]); ax.set_xlabel("Space type"); ax.set_ylabel("Model")
fig.tight_layout()
save_fig(fig, "figure06_heatmaps")

In [ ]:
# 8. Two-way factorial analysis (model x space type)
twoway_full, art_full, assumption_rows, summary_rows, resid_store = [], [], [], [], {}
for dv in DVS:
    res = twoway_anova(master_df, dv, "model", "space_type")
    tab = res["table"].assign(dv=dv)
    resid_store[dv] = res["residuals"]
    sw = stats.shapiro(res["residuals"])
    cells = [g[dv].dropna().to_numpy(float) for _, g in master_df.groupby(["model", "space_type"], observed=True)]
    bf = stats.levene(*cells, center="median")
    primary = "parametric" if (sw.pvalue > ALPHA and bf.pvalue > ALPHA) else "ART"
    art = art_anova(master_df, dv, "model", "space_type").assign(dv=dv)
    twoway_full.append(tab)
    art_full.append(art)
    assumption_rows.append({"dv": dv, "residual_shapiro_W": sw.statistic, "residual_shapiro_p": sw.pvalue,
                            "brown_forsythe_F": bf.statistic, "brown_forsythe_p": bf.pvalue,
                            "primary_test": primary})
    for _, r in tab[tab.effect != "Residual"].iterrows():
        a = art.loc[art.effect == r["effect"]].iloc[0]
        summary_rows.append({
            "dv": dv, "effect": r["effect"], "primary_test": primary,
            "F": r["F"], "df": r["df"], "df_error": r["df_error"], "p": r["p"],
            "partial_eta2": r["partial_eta2"], "partial_omega2": r["partial_omega2"],
            "achieved_power": r["achieved_power"],
            "ART_F": a["F"], "ART_df": a["df"], "ART_df_error": a["df_error"], "ART_p": a["p"],
            "p_primary": r["p"] if primary == "parametric" else a["p"],
        })

twoway_full_df = front(pd.concat(twoway_full, ignore_index=True), ["dv"])
art_full_df = front(pd.concat(art_full, ignore_index=True), ["dv"])
assumptions_df = pd.DataFrame(assumption_rows)
twoway_summary_df = pd.DataFrame(summary_rows)
twoway_summary_df["p_holm"] = np.nan
for eff in twoway_summary_df.effect.unique():
    idx = twoway_summary_df.effect == eff
    twoway_summary_df.loc[idx, "p_holm"] = holm(twoway_summary_df.loc[idx, "p_primary"])
twoway_summary_df["significant"] = twoway_summary_df["p_holm"] < ALPHA
twoway_summary_df.insert(2, "effect_label", twoway_summary_df.effect.map(EFFECT_LABELS))

display(assumptions_df.round(4))
display(twoway_summary_df.round(4))

fig, axes = plt.subplots(1, len(DVS), figsize=(4.2 * len(DVS), 4))
for ax, dv in zip(axes, DVS):
    stats.probplot(resid_store[dv], dist="norm", plot=ax)
    ax.set_title(DV_LABELS[dv], fontsize=10); ax.set_xlabel("Theoretical quantiles"); ax.set_ylabel("Residuals")
fig.tight_layout()
save_fig(fig, "figure04_residual_qq_plots")

In [ ]:
# 9. RQ1: comparison of generative models
inter_sig = dict(twoway_summary_df.loc[twoway_summary_df.effect == "model:space_type", ["dv", "significant"]].values)
PH_COLS = ["dv", "space_type", "method", "A", "B", "p_adj", "effect_size", "effect_size_type"]

rq1_df = pd.DataFrame([oneway_tests(master_df, dv, "model", model_levels) for dv in DVS])
rq1_df["p_primary_holm"] = holm(rq1_df["p_primary"])
rq1_posthoc = []
for _, r in rq1_df.iterrows():
    if r["p_primary_holm"] < ALPHA:
        rq1_posthoc.append(posthoc(master_df, r["dv"], "model", model_levels, r["primary_test"]).assign(dv=r["dv"]))
rq1_posthoc_df = front(pd.concat(rq1_posthoc, ignore_index=True), PH_COLS) if rq1_posthoc else pd.DataFrame()

simple_rows, simple_ph = [], []
for dv in DVS:
    if not inter_sig.get(dv, False):
        continue
    part = []
    for sp in space_levels:
        r = oneway_tests(master_df[master_df.space_type == sp], dv, "model", model_levels)
        if r:
            r["space_type"] = sp
            part.append(r)
    part = pd.DataFrame(part)
    part["p_primary_holm"] = holm(part["p_primary"])
    simple_rows.append(part)
    for _, r in part[part.p_primary_holm < ALPHA].iterrows():
        sub = master_df[master_df.space_type == r["space_type"]]
        simple_ph.append(posthoc(sub, dv, "model", model_levels, r["primary_test"])
                         .assign(dv=dv, space_type=r["space_type"]))
rq1_simple_df = front(pd.concat(simple_rows, ignore_index=True), ["dv", "space_type"]) if simple_rows else pd.DataFrame()
rq1_simple_posthoc_df = front(pd.concat(simple_ph, ignore_index=True), PH_COLS) if simple_ph else pd.DataFrame()

display(rq1_df[["dv", "primary_test", "p_primary", "p_primary_holm", "eta2", "omega2",
                "epsilon2", "achieved_power"]].round(4))
if len(rq1_simple_df):
    display(rq1_simple_df[["dv", "space_type", "primary_test", "p_primary", "p_primary_holm", "eta2"]].round(4))

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, dv in zip(axes.flat, DVS):
    sns.boxplot(data=master_df, x="model_label", y=dv, order=MODEL_LABEL_ORDER, palette=MODEL_PALETTE,
                hue="model_label", legend=False, showfliers=False, ax=ax)
    sns.stripplot(data=master_df, x="model_label", y=dv, order=MODEL_LABEL_ORDER, color="black",
                  alpha=0.35, size=2.5, jitter=0.2, ax=ax)
    ax.set_title(DV_LABELS[dv]); ax.set_xlabel(""); ax.set_ylabel(DV_LABELS[dv])
    ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
save_fig(fig, "figure07_model_comparison")

In [ ]:
# 10. RQ2: comparison of space types
rq2_df = pd.DataFrame([oneway_tests(master_df, dv, "space_type", space_levels) for dv in DVS])
rq2_df["p_primary_holm"] = holm(rq2_df["p_primary"])
rq2_posthoc = []
for _, r in rq2_df.iterrows():
    if r["p_primary_holm"] < ALPHA:
        rq2_posthoc.append(posthoc(master_df, r["dv"], "space_type", space_levels, r["primary_test"]).assign(dv=r["dv"]))
rq2_posthoc_df = front(pd.concat(rq2_posthoc, ignore_index=True), PH_COLS) if rq2_posthoc else pd.DataFrame()

rq2_contrast_df = pd.DataFrame()
if {"corridor", "lounge"} <= set(space_levels):
    xc = master_df.loc[master_df.space_type == "corridor", PRIMARY_D].dropna().to_numpy(float)
    xl = master_df.loc[master_df.space_type == "lounge", PRIMARY_D].dropna().to_numpy(float)
    tt = stats.ttest_ind(xc, xl, equal_var=False, alternative="less")
    mw = stats.mannwhitneyu(xc, xl, alternative="less")
    va, vb = xc.var(ddof=1) / len(xc), xl.var(ddof=1) / len(xl)
    df_w = (va + vb) ** 2 / (va ** 2 / (len(xc) - 1) + vb ** 2 / (len(xl) - 1))
    diff = xc.mean() - xl.mean()
    tq = stats.t.ppf(0.975, df_w)
    rq2_contrast_df = pd.DataFrame([{
        "dv": PRIMARY_D, "contrast": "corridor < lounge (one-sided)",
        "n_corridor": len(xc), "n_lounge": len(xl), "mean_corridor": xc.mean(), "mean_lounge": xl.mean(),
        "mean_diff": diff, "diff_ci95_lower": diff - tq * np.sqrt(va + vb), "diff_ci95_upper": diff + tq * np.sqrt(va + vb),
        "welch_t": tt.statistic, "welch_df": df_w, "welch_p_one_sided": tt.pvalue,
        "hedges_g": hedges_g(xc, xl),
        "mannwhitney_U": mw.statistic, "mannwhitney_p_one_sided": mw.pvalue,
        "rank_biserial_r": 2 * mw.statistic / (len(xc) * len(xl)) - 1,
    }])

display(rq2_df[["dv", "primary_test", "p_primary", "p_primary_holm", "eta2", "omega2",
                "epsilon2", "achieved_power"]].round(4))
if len(rq2_contrast_df):
    display(rq2_contrast_df.round(4))

fig, axes = plt.subplots(1, len(DVS), figsize=(4.3 * len(DVS), 4.6))
for ax, dv in zip(axes, DVS):
    sns.pointplot(data=master_df, x="space_label", y=dv, order=SPACE_LABEL_ORDER, errorbar=("ci", 95),
                  seed=RANDOM_SEED, color="#4C72B0", capsize=0.15, ax=ax)
    ax.set_title(DV_LABELS[dv], fontsize=11); ax.set_xlabel("Space type"); ax.set_ylabel(DV_LABELS[dv])
fig.tight_layout()
save_fig(fig, "figure08_space_type_comparison")

In [ ]:
# 11. RQ3: sensitivity to the thresholding method
pair = master_df[["image_id", "model", "space_type", "D_otsu", "D_adaptive"]].dropna()
diffs = (pair["D_otsu"] - pair["D_adaptive"]).to_numpy()
n_pair = len(diffs)
if stats.shapiro(diffs).pvalue > ALPHA:
    tr = stats.ttest_rel(pair["D_otsu"], pair["D_adaptive"])
    test_name, stat_val, p_val = "paired t-test", float(tr.statistic), float(tr.pvalue)
else:
    wr = stats.wilcoxon(pair["D_otsu"], pair["D_adaptive"])
    test_name, stat_val, p_val = "Wilcoxon signed-rank", float(wr.statistic), float(wr.pvalue)
dz = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else np.nan
loa_low, loa_high = diffs.mean() - 1.96 * diffs.std(ddof=1), diffs.mean() + 1.96 * diffs.std(ddof=1)
rq3_icc_df = icc_table(pair[["D_otsu", "D_adaptive"]].to_numpy())
r_img = corr_with_ci(pair["D_otsu"], pair["D_adaptive"], method="spearman")

rq3_agreement_df = pd.DataFrame([{
    "n_images": n_pair, "test": test_name, "statistic": stat_val, "p": p_val,
    "mean_D_otsu": pair["D_otsu"].mean(), "mean_D_adaptive": pair["D_adaptive"].mean(),
    "mean_difference": diffs.mean(), "sd_difference": diffs.std(ddof=1), "cohens_dz": dz,
    "achieved_power": power_paired_t(dz, n_pair),
    "bland_altman_loa_lower": loa_low, "bland_altman_loa_upper": loa_high,
    "share_within_loa": float(np.mean((diffs >= loa_low) & (diffs <= loa_high))),
    "spearman_rho_image_level": r_img["r"], "spearman_ci95_lower": r_img["ci95_lower"],
    "spearman_ci95_upper": r_img["ci95_upper"],
    "icc21_absolute_agreement": rq3_icc_df.loc[rq3_icc_df.type == "ICC(2,1)", "icc"].iloc[0],
    "icc31_consistency": rq3_icc_df.loc[rq3_icc_df.type == "ICC(3,1)", "icc"].iloc[0],
}])

long = pair.melt(id_vars=["image_id", "model", "space_type"], value_vars=["D_otsu", "D_adaptive"],
                 var_name="method", value_name="D")
grand = long["D"].mean()
ss_total = ((long["D"] - grand) ** 2).sum()
decomp = []
for factor in ("method", "model", "space_type", "image_id"):
    means = long.groupby(factor, observed=True)["D"].transform("mean")
    decomp.append({"source": factor, "SS": float(((means - grand) ** 2).sum())})
rq3_variance_df = pd.DataFrame(decomp)
rq3_variance_df["eta2_of_total"] = rq3_variance_df["SS"] / ss_total
rq3_variance_df.loc[rq3_variance_df.source == "image_id", "source"] = "image (incl. model and space)"

robust_rows = []
for dv in ("D_otsu", "D_adaptive"):
    tab = twoway_anova(master_df, dv, "model", "space_type")["table"]
    art = art_anova(master_df, dv, "model", "space_type")
    for _, r in tab[tab.effect != "Residual"].iterrows():
        a = art.loc[art.effect == r["effect"]].iloc[0]
        robust_rows.append({"dv": dv, "effect": r["effect"], "F": r["F"], "p": r["p"],
                            "partial_eta2": r["partial_eta2"], "ART_F": a["F"], "ART_p": a["p"],
                            "significant_param": r["p"] < ALPHA, "significant_ART": a["p"] < ALPHA})
rq3_robustness_df = pd.DataFrame(robust_rows)
rank_otsu = master_df.groupby("model", observed=True)["D_otsu"].mean().rank()
rank_adap = master_df.groupby("model", observed=True)["D_adaptive"].mean().rank()
rank_rho = stats.spearmanr(rank_otsu, rank_adap)[0] if len(rank_otsu) > 2 else np.nan
rq3_model_ranks_df = pd.DataFrame({"model": [str(m) for m in rank_otsu.index],
                                   "model_label": [model_label(str(m)) for m in rank_otsu.index],
                                   "rank_D_otsu": rank_otsu.values, "rank_D_adaptive": rank_adap.values})
rq3_model_ranks_df["spearman_rho_model_ranks"] = rank_rho

display(rq3_agreement_df.T.round(4))
display(rq3_icc_df.round(3))
display(rq3_variance_df.round(4))
display(rq3_robustness_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for m in model_levels:
    s = pair[pair.model == m]
    axes[0].scatter(s["D_otsu"], s["D_adaptive"], s=16, alpha=0.7, label=model_label(m),
                    color=MODEL_PALETTE.get(model_label(m)))
lims = [min(pair.D_otsu.min(), pair.D_adaptive.min()), max(pair.D_otsu.max(), pair.D_adaptive.max())]
axes[0].plot(lims, lims, "k--", lw=1, label="Identity line")
axes[0].set_xlabel(DV_LABELS["D_otsu"]); axes[0].set_ylabel(DV_LABELS["D_adaptive"])
axes[0].set_title(f"Agreement between thresholding methods (Spearman ρ = {r_img['r']:.2f})")
axes[0].legend(fontsize=8)
mean_D = (pair["D_otsu"] + pair["D_adaptive"]) / 2
axes[1].scatter(mean_D, diffs, s=16, alpha=0.6)
axes[1].axhline(diffs.mean(), color="red", ls="--", label=f"Mean difference = {diffs.mean():.3f}")
axes[1].axhline(loa_low, color="gray", ls=":")
axes[1].axhline(loa_high, color="gray", ls=":", label="95% limits of agreement")
axes[1].set_xlabel("Mean of the two methods"); axes[1].set_ylabel("D_otsu − D_adaptive")
axes[1].set_title("Bland–Altman plot"); axes[1].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "figure09_threshold_sensitivity")

In [ ]:
# 12. Correlations between metrics
corr_vars = [c for c in ALL_METRICS if master_df[c].notna().sum() > 3]
corr_matrix = master_df[corr_vars].corr(method="spearman")
pairs = []
for a, b in itertools.combinations(corr_vars, 2):
    pairs.append({"variable_1": a, "variable_2": b, **corr_with_ci(master_df[a], master_df[b], method="spearman")})
corr_pairs_df = pd.DataFrame(pairs)
corr_pairs_df["p_holm"] = holm(corr_pairs_df["p"])

fig, ax = plt.subplots(figsize=(1.0 * len(corr_vars) + 2, 0.9 * len(corr_vars) + 1.5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=ax, cbar_kws={"label": "Spearman's ρ"},
            xticklabels=[DV_LABELS.get(c, c) for c in corr_vars],
            yticklabels=[DV_LABELS.get(c, c) for c in corr_vars])
ax.set_title("Spearman correlations among image metrics")
ax.grid(False)
fig.tight_layout()
save_fig(fig, "figure03_correlation_matrix")

In [ ]:
# 13. Reproducibility information, results summary and Excel report
import skimage as _skimage
try:
    import timm as _timm
    timm_version = _timm.__version__
except Exception:
    timm_version = "n/a"

repro_df = pd.DataFrame(list({
    "run_date": datetime.now().date().isoformat(),
    "python": sys.version.split()[0],
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
    "scikit_image": _skimage.__version__, "opencv": cv2.__version__,
    "seaborn": sns.__version__, "matplotlib": matplotlib.__version__, "torch": torch.__version__,
    "timm": timm_version, "device": DEVICE, "midas_model": MIDAS_MODEL_TYPE,
    "target_image_size_px": TARGET_SIZE, "box_sizes_px": ", ".join(map(str, BOX_SIZES)),
    "threshold_methods": ", ".join(THRESH_METHODS), "adaptive_block_size": ADAPTIVE_BLOCK_SIZE,
    "primary_fractal_metric": PRIMARY_D, "alpha": ALPHA, "random_seed": RANDOM_SEED,
    "n_images": len(master_df), "n_models": n_models, "n_space_types": n_spaces,
}.items()), columns=["parameter", "value"])

def apa_twoway(r):
    s = (f"F({r['df']:.0f}, {r['df_error']:.0f}) = {r['F']:.2f}, {fmt_p(r['p'])}, "
         f"ηp² = {r['partial_eta2']:.3f}, ωp² = {r['partial_omega2']:.3f}; "
         f"ART: F({r['ART_df']:.0f}, {r['ART_df_error']:.0f}) = {r['ART_F']:.2f}, {fmt_p(r['ART_p'])}")
    return s + f"; Holm-adjusted p (primary test = {r['primary_test']}) " + fmt_p(r["p_holm"]).replace("p ", "", 1)

def apa_oneway(r):
    if r["primary_test"] == "welch_anova":
        s = f"Welch F({r['welch_df1']:.0f}, {r['welch_df2']:.2f}) = {r['welch_F']:.2f}, {fmt_p(r['welch_p'])}, ω² = {r['omega2']:.3f}"
    else:
        s = f"H({r['kruskal_df']:.0f}) = {r['kruskal_H']:.2f}, {fmt_p(r['kruskal_p'])}, ε² = {r['epsilon2']:.3f}"
    return s + "; Holm " + fmt_p(r["p_primary_holm"])

lines = [f"Design: {n_models} models × {n_spaces} space types; N = {N_TOTAL} images.", ""]
lines.append("Two-way analyses (model × space type):")
for _, r in twoway_summary_df.iterrows():
    lines.append(f"  {DV_LABELS[r['dv']]} - {EFFECT_LABELS[r['effect']]}: {apa_twoway(r)}")
lines += ["", "RQ1 - one-way comparisons across models (space types pooled):"]
for _, r in rq1_df.iterrows():
    lines.append(f"  {DV_LABELS[r['dv']]}: {apa_oneway(r)}")
lines += ["", "RQ2 - one-way comparisons across space types (models pooled):"]
for _, r in rq2_df.iterrows():
    lines.append(f"  {DV_LABELS[r['dv']]}: {apa_oneway(r)}")
if len(rq2_contrast_df):
    c = rq2_contrast_df.iloc[0]
    lines.append(f"  H2 planned contrast (corridor < lounge, {DV_LABELS[c['dv']]}): "
                 f"M = {c['mean_corridor']:.3f} vs {c['mean_lounge']:.3f}, Welch t({c['welch_df']:.2f}) = {c['welch_t']:.2f}, "
                 f"one-sided {fmt_p(c['welch_p_one_sided'])}, g = {c['hedges_g']:.2f}; "
                 f"Mann-Whitney U = {c['mannwhitney_U']:.0f}, one-sided {fmt_p(c['mannwhitney_p_one_sided'])}")
a = rq3_agreement_df.iloc[0]
lines += ["", "RQ3 - thresholding sensitivity:",
          f"  {a['test']}: statistic = {a['statistic']:.2f}, {fmt_p(a['p'])}, mean difference = {a['mean_difference']:.3f} "
          f"(SD = {a['sd_difference']:.3f}), d_z = {a['cohens_dz']:.2f}; 95% limits of agreement [{a['bland_altman_loa_lower']:.3f}, "
          f"{a['bland_altman_loa_upper']:.3f}]; ICC(2,1) = {a['icc21_absolute_agreement']:.2f}, ICC(3,1) = {a['icc31_consistency']:.2f}; "
          f"image-level Spearman ρ = {a['spearman_rho_image_level']:.2f}."]
lines += ["", "Sensitivity power analysis (80% power, α = .05):"]
for _, r in power_sensitivity_df[power_sensitivity_df.power == 0.80].iterrows():
    lines.append(f"  {r['effect']}: minimum detectable f = {r['min_detectable_cohens_f']:.3f} "
                 f"(ηp² = {r['min_detectable_partial_eta2']:.3f}) with N = {r['N']:.0f}")
apa_text = "\n".join(lines)
(DIRS["tables"] / "results_summary.txt").write_text(apa_text, encoding="utf-8")
print(apa_text)
apa_df = pd.DataFrame({"line": lines})

REPORT_PATH = DIRS["tables"] / "full_statistical_report.xlsx"
sheets = {
    "01_design_counts": design_counts, "02_design_replicates": design_reps,
    "03_design_file_format": design_format, "04_design_resolution": design_resolution,
    "05_image_level_data": master_df,
    "06_desc_by_model": desc_model, "07_desc_by_space": desc_space, "08_desc_model_x_space": desc_cell,
    "09_power_apriori": power_apriori_df, "10_power_sensitivity": power_sensitivity_df,
    "11_assumptions_twoway": assumptions_df, "12_twoway_summary": twoway_summary_df,
    "13_twoway_full_tables": twoway_full_df, "14_ART_full_tables": art_full_df,
    "15_RQ1_oneway": rq1_df, "16_RQ1_posthoc": rq1_posthoc_df,
    "17_RQ1_simple_effects": rq1_simple_df, "18_RQ1_simple_posthoc": rq1_simple_posthoc_df,
    "19_RQ2_oneway": rq2_df, "20_RQ2_posthoc": rq2_posthoc_df, "21_RQ2_planned_contrast": rq2_contrast_df,
    "22_RQ3_agreement": rq3_agreement_df, "23_RQ3_icc": rq3_icc_df, "24_RQ3_variance": rq3_variance_df,
    "25_RQ3_robustness": rq3_robustness_df, "26_RQ3_model_ranks": rq3_model_ranks_df,
    "27_correlations_pairs": corr_pairs_df, "28_correlation_matrix": corr_matrix.reset_index(names="variable"),
    "29_results_summary": apa_df, "30_reproducibility": repro_df,
}
write_excel(sheets, REPORT_PATH)
repro_df.to_csv(DIRS["tables"] / "reproducibility_info.csv", index=False)
print(f"\nReport saved: {REPORT_PATH}")

In [ ]:
# 14. Export outputs
zip_base = BASE_DIR / "visual_comfort_outputs"
shutil.make_archive(str(zip_base), "zip", str(BASE_DIR / "outputs"))
zip_path = Path(str(zip_base) + ".zip")
print(f"Archive: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    files.download(str(zip_path))